# 01 — Entraînement ModCloth Fit Model V2

**But :** entraîner et évaluer le modèle TensorFlow/Keras de prédiction de fit (`small`, `fit`, `large`) à partir du dataset ModCloth.

Ce notebook exécute uniquement la V2 du pipeline ModCloth :
- téléchargement Kaggle ;
- inspection des données ;
- entraînement du MLP ;
- métriques et artefacts versionnés ;
- sauvegarde finale dans Google Drive.

Il ne traite ni Fashion Product Images Small, ni Polyvore.

> **Important — Secrets Colab**
>
> - Ton secret Kaggle s’appelle **`KAGGLE_API`**. Le notebook le lit sous ce nom, puis le transmet à Kaggle via la variable d’environnement requise.
> - `GITHUB_TOKEN` est facultatif et nécessaire seulement si ton dépôt GitHub est privé.


## 1. Monter Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Cloner ou mettre à jour le repo GitHub

Le code du projet reste dans GitHub. Colab clone une copie temporaire dans `/content`.

- Si le repo est public : aucune information supplémentaire n’est nécessaire.
- S’il est privé : crée un Secret Colab optionnel `GITHUB_TOKEN` avec un token GitHub ayant seulement l’accès lecture au contenu du repo.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

REPO_URL = "https://github.com/MilFhey/fit-outfit-advisor.git"
REPO_DIR = Path("/content/fit-outfit-advisor")
BRANCH = "main"


def get_optional_secret(name: str) -> str | None:
    try:
        value = userdata.get(name)
    except Exception:
        return None
    return value or None


def build_git_environment(github_token: str | None) -> tuple[dict[str, str], Path | None]:
    """Retourne un environnement Git ; utilise GIT_ASKPASS seulement si le repo est privé."""
    env = os.environ.copy()

    if not github_token:
        return env, None

    askpass_file = Path("/tmp/git_askpass.sh")
    askpass_file.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
        encoding="utf-8",
    )
    askpass_file.chmod(0o700)

    env["GITHUB_TOKEN"] = github_token
    env["GIT_ASKPASS"] = str(askpass_file)
    env["GIT_TERMINAL_PROMPT"] = "0"

    return env, askpass_file


github_token = get_optional_secret("GITHUB_TOKEN")
git_env, askpass_file = build_git_environment(github_token)

try:
    # Supprime uniquement un dossier incomplet provenant d’un clone interrompu.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists():
        print(f"Repo déjà présent, mise à jour : {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], env=git_env, check=True)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
            env=git_env,
            check=True,
        )
    else:
        print(f"Clonage du repo : {REPO_URL}")
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            env=git_env,
            check=True,
        )
finally:
    if askpass_file is not None:
        askpass_file.unlink(missing_ok=True)

# Le repo peut contenir le projet directement à sa racine ou dans un dossier imbriqué.
candidate_project_dirs = [
    REPO_DIR,
    REPO_DIR / "fit-outfit-advisor",
]

PROJECT_DIR = next(
    (candidate for candidate in candidate_project_dirs if (candidate / "src").is_dir()),
    None,
)

if PROJECT_DIR is None:
    repo_contents = [path.name for path in REPO_DIR.iterdir()] if REPO_DIR.exists() else []
    raise FileNotFoundError(
        "Impossible de localiser le dossier src du projet. "
        f"Contenu de {REPO_DIR} : {repo_contents}"
    )

os.chdir(PROJECT_DIR)

print(f"Repo cloné : {REPO_DIR}")
print(f"Projet détecté : {PROJECT_DIR}")
print(f"Répertoire courant : {Path.cwd()}")


## 3. Installer les dépendances


In [ ]:
requirements_path = PROJECT_DIR / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(f"requirements.txt absent : {requirements_path}")

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_path)],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "kaggle"], check=True)

print("✅ Dépendances installées.")


## 4. Créer les dossiers temporaires Colab


In [ ]:
RUNTIME_ROOT = Path("/content/fit-outfit-runtime")
KAGGLE_DOWNLOAD_DIR = RUNTIME_ROOT / "kaggle_downloads"
CONTENT_DATA_DIR = RUNTIME_ROOT / "data"
CONTENT_ARTIFACT_DIR = RUNTIME_ROOT / "artifacts"

for directory in [RUNTIME_ROOT, KAGGLE_DOWNLOAD_DIR, CONTENT_DATA_DIR, CONTENT_ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(directory)


## 5. Charger le secret Kaggle

Ton Secret Colab s’appelle **`KAGGLE_API`**.

Sa valeur doit être uniquement ton token Kaggle moderne, qui commence par `KGAT_`.

Le code le lit sous le nom `KAGGLE_API`, puis le place dans `KAGGLE_API_TOKEN` pour que la commande `kaggle` puisse s’authentifier.


In [ ]:
try:
    kaggle_token = userdata.get("KAGGLE_API")
except Exception as exc:
    raise ValueError(
        "Secret Colab introuvable : crée ou autorise le Secret nommé exactement KAGGLE_API."
    ) from exc

if not kaggle_token or not kaggle_token.startswith("KGAT_"):
    raise ValueError(
        "Le Secret KAGGLE_API est absent ou invalide. "
        "Il doit contenir uniquement un token Kaggle moderne commençant par KGAT_."
    )

# Kaggle CLI attend cette variable d’environnement ; le Secret Colab peut conserver ton nom KAGGLE_API.
os.environ["KAGGLE_API_TOKEN"] = kaggle_token
os.environ["KAGGLE_API"] = kaggle_token

print("✅ Token Kaggle chargé depuis le Secret Colab KAGGLE_API.")


## 6. Tester l’accès à Kaggle


In [ ]:
result = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "clothing fit dataset"],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "Échec d’authentification ou de connexion Kaggle.\n"
        f"Erreur : {result.stderr}"
    )

print(result.stdout[:2000])


## 7. Télécharger le dataset ModCloth


In [ ]:
KAGGLE_DATASET = "rmisra/clothing-fit-dataset-for-size-recommendation"
FORCE_DOWNLOAD = False

existing_files = [path for path in KAGGLE_DOWNLOAD_DIR.rglob("*") if path.is_file()]

if FORCE_DOWNLOAD or not existing_files:
    command = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        KAGGLE_DATASET,
        "-p",
        str(KAGGLE_DOWNLOAD_DIR),
        "--unzip",
    ]
    subprocess.run(command, check=True)
else:
    print("Téléchargement déjà présent : réutilisation des fichiers existants.")

downloaded_files = sorted(path for path in KAGGLE_DOWNLOAD_DIR.rglob("*") if path.is_file())

if not downloaded_files:
    raise FileNotFoundError(
        "Aucun fichier téléchargé depuis Kaggle. Vérifie le token et le slug du dataset."
    )

for path in downloaded_files:
    print(path)


## 8. Détecter le fichier ModCloth

Le dataset peut être fourni en CSV, JSON ou JSONL.  
S’il est en JSON/JSONL, le notebook génère un CSV temporaire dans l’espace Colab.


In [ ]:
import pandas as pd

csv_files = sorted(KAGGLE_DOWNLOAD_DIR.rglob("*.csv"))
modcloth_csv_files = [path for path in csv_files if "modcloth" in path.name.lower()]

if modcloth_csv_files:
    DATASET_PATH = modcloth_csv_files[0]
    print(f"CSV ModCloth détecté : {DATASET_PATH}")
elif csv_files:
    DATASET_PATH = csv_files[0]
    print(f"CSV détecté : {DATASET_PATH}")
else:
    json_files = sorted(
        [*KAGGLE_DOWNLOAD_DIR.rglob("*.json"), *KAGGLE_DOWNLOAD_DIR.rglob("*.jsonl")]
    )
    modcloth_json_files = [path for path in json_files if "modcloth" in path.name.lower()]

    if not modcloth_json_files:
        raise FileNotFoundError(
            "Aucun CSV, JSON ou JSONL ModCloth détecté dans le téléchargement Kaggle."
        )

    source_json = modcloth_json_files[0]
    print(f"JSON/JSONL ModCloth détecté : {source_json}")

    try:
        df_json = pd.read_json(source_json, lines=True)
    except ValueError:
        df_json = pd.read_json(source_json)

    DATASET_PATH = CONTENT_DATA_DIR / "modcloth_final_data.csv"
    df_json.to_csv(DATASET_PATH, index=False)
    print(f"CSV temporaire généré : {DATASET_PATH}")

print(f"DATASET_PATH = {DATASET_PATH}")


## 9. Inspecter le dataset avant entraînement


In [ ]:
df = pd.read_csv(DATASET_PATH, low_memory=False)

print("df.shape =", df.shape)

print("\nColonnes :")
print(list(df.columns))

print("\nTypes :")
display(df.dtypes.to_frame("dtype"))

print("\nAperçu :")
display(df.head())

missing_values = df.isna().sum().sort_values(ascending=False)
print("\nValeurs manquantes par colonne :")
display(missing_values.to_frame("missing_count"))


## 10. Lancer l’entraînement ModCloth V2

Cette cellule lance le vrai script du repo.  
Le script doit créer les artefacts versionnés dans `models/fit_v2/`.

L’entraînement ne démarre que si :
- le dataset est présent ;
- le dossier projet est détecté ;
- `src/training/train_fit_model.py` existe.


In [ ]:
EPOCHS = 20
BATCH_SIZE = 64

training_script = PROJECT_DIR / "src" / "training" / "train_fit_model.py"

if not Path(DATASET_PATH).exists():
    raise FileNotFoundError(f"Dataset absent : {DATASET_PATH}")

if not training_script.exists():
    raise FileNotFoundError(f"Script d’entraînement absent : {training_script}")

training_env = os.environ.copy()
training_env["PYTHONPATH"] = (
    f"{PROJECT_DIR}{os.pathsep}{training_env.get('PYTHONPATH', '')}".rstrip(os.pathsep)
)

print(f"Répertoire courant : {PROJECT_DIR}")
print(f"Script exécuté : {training_script}")
print(f"Dataset : {DATASET_PATH}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "src.training.train_fit_model",
        "--dataset",
        str(DATASET_PATH),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
    ],
    cwd=str(PROJECT_DIR),
    env=training_env,
    check=True,
)


## 11. Vérifier les métriques et artefacts V2


In [ ]:
import json

FIT_V2_DIR = PROJECT_DIR / "models" / "fit_v2"

artifact_paths = [
    FIT_V2_DIR / "fit_model.keras",
    FIT_V2_DIR / "fit_preprocessor.joblib",
    FIT_V2_DIR / "fit_label_encoder.joblib",
    FIT_V2_DIR / "metadata.json",
    FIT_V2_DIR / "metrics.json",
    FIT_V2_DIR / "confusion_matrix_raw.png",
    FIT_V2_DIR / "confusion_matrix_normalized.png",
    FIT_V2_DIR / "training_history.png",
]

print("Artefacts attendus :")
for path in artifact_paths:
    status = "OK" if path.exists() else "ABSENT"
    size = path.stat().st_size if path.exists() else 0
    print(f"{status:6} {size:>12} bytes  {path}")

missing_artifacts = [path for path in artifact_paths if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(
        "Artefacts manquants après entraînement : "
        + ", ".join(str(path) for path in missing_artifacts)
    )

metadata_path = FIT_V2_DIR / "metadata.json"
metrics_path = FIT_V2_DIR / "metrics.json"

print("\nMetadata fit_v2 :")
print(metadata_path.read_text(encoding="utf-8"))

print("\nMetrics fit_v2 :")
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))


## 12. Copier les artefacts V2 vers Google Drive


In [ ]:
DRIVE_ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/fit-outfit-advisor/artifacts/modcloth_fit_v2"
)
DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for path in artifact_paths:
    destination = DRIVE_ARTIFACT_DIR / path.name
    shutil.copy2(path, destination)
    print(f"Copie : {path.name} -> {destination}")

print(f"\n✅ Artefacts disponibles dans Google Drive : {DRIVE_ARTIFACT_DIR}")
